In [ ]:
# 1. Könyvtárak importálása
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # Ez a sor megoldja az "Intel OpenMP" könyvtár duplikációs hibáját, ami néha előfordulhat bizonyos gépeken és könyvtárkombinációknál.

# 2. EZ A KULCS: Környezeti változóként mondjuk meg a Pythonnak, hogy a háttérfolyamatokban is némítsa el ezt a specifikus figyelmeztetést
os.environ["PYTHONWARNINGS"] = "ignore:Found Intel OpenMP:RuntimeWarning"
# Biztonsági öv a fő folyamatnak is
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="threadpoolctl")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, TweedieRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.svm import SVR
import joblib


# 2. Fájlbeolvasás és alap statisztikák
df1 = pd.read_csv('../data/raw/CPU_benchmark_v4.csv')

print(df1.info())

print(f"Az adathalmaz teljes mérete: {df1.shape[0]} sor és {df1.shape[1]} oszlop")
print(f"Sorok száma hiányzó értékkel: {df1.isnull().any(axis=1).sum()}")

def get_stats(df, name):
    # Hiányzó értékek (NaN)
    null_counts = df.isnull().sum()
    # Nulla értékek (csak numerikus oszlopoknál releváns)
    zero_counts = (df == 0).sum()

    stats = pd.DataFrame({
        'Hiányzó (NaN)': null_counts,
        'Nulla érték (0)': zero_counts
    })
    print(f"\n--- {name} statisztikák ---")
    print(stats)
    return stats

# Megjelen#t3s
stats1 = get_stats(df1, "CPU_benchmark_v4")


hianyzikTDPandPrice = (df1['TDP'].isna() & df1['price'].isna()).sum()
print(f"\nPrice és TDP mező hiányzik: {hianyzikTDPandPrice}")
darab = (df1['TDP'].isna() & df1['price'].notna()).sum()
print(f"Price van, de TDP hiányzik: {darab}")

df_clean = df1.dropna(subset=['TDP'])
stats2 = get_stats(df_clean, "CPU_benchmark_v4")

# 3. GLOBÁLIS ADATTISZTÍTÁS (Csak egyszer fut le!)
def clean_numeric(val):
    if isinstance(val, str):
        return float(val.replace(',', ''))
    return val

# A powerPerf, price, stb. konverziója stringből floattá a teljes df_clean-en
numeric_cols = ['price', 'cpuMark', 'threadMark', 'TDP', 'powerPerf', 'cores']
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(clean_numeric)

# 4. GLOBÁLIS VÁLTOZÓK (Socket csoportosítás)
top_20_sockets = df_clean['socket'].value_counts().nlargest(20).index
df_clean['socket_grouped'] = df_clean['socket'].apply(lambda x: x if x in top_20_sockets else 'Other')

# 5. FEATURE LISTÁK
# A) Korrelációs analízishez (itt vizsgálni akarjuk a powerPerf-et is)
corr_features = ['price', 'cpuMark', 'threadMark', 'TDP', 'powerPerf', 'cores', 'testDate', 'category']

# C) Ár (price) modellezéséhez és pótlásához (itt a célváltozók és deriváltak nincsenek bent)
features_price = ['cpuMark', 'threadMark', 'TDP', 'cores', 'testDate', 'category', 'socket_grouped']

# D) Végső Power Performance predikcióhoz (itt az ár már fontos bemeneti változó)
features_pp = ['price', 'threadMark', 'cores', 'testDate', 'category', 'socket_grouped']

In [ ]:
# === Korreláció-analízis ===
# Csak azokat az oszlopokat tartjuk meg, amik ténylegesen benne vannak a df_clean-ben
existing_features = [f for f in corr_features if f in df_clean.columns]
df_analysis_clean = df_clean[existing_features].copy()

# One-Hot Encoding alkalmazása
df_corr_analysis = pd.get_dummies(df_analysis_clean, columns=['category'] if 'category' in df_analysis_clean.columns else [], prefix='cat')

# Csak a numerikus oszlopok kiválasztása
numeric_df = df_corr_analysis.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature-ök korrelációja')
plt.tight_layout()
plt.show()

In [ ]:
# Csak azokat a sorokat nézzük, ahol van ár
df_price_analysis = df_clean.dropna(subset=['price']).copy()

# 1. Ár eloszlása Category szerint
plt.figure(figsize=(14, 7))
sns.boxplot(data=df_price_analysis, x='price', y='category', hue='category', palette='Set2', legend=False)
plt.title('Ár eloszlása kategóriánként')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Átlagos árak kategóriánként:")
display(df_price_analysis.groupby('category')['price'].mean().sort_values(ascending=False))

# 2. Ár eloszlása a leggyakoribb Socket-ek szerint (Top 15)
top_sockets = df_price_analysis['socket'].value_counts().nlargest(15).index
df_top_sockets_price = df_price_analysis[df_price_analysis['socket'].isin(top_sockets)]

plt.figure(figsize=(14, 7))
sns.boxplot(data=df_top_sockets_price, x='price', y='socket', hue='socket', palette='viridis', legend=False)
plt.title('Ár eloszlása a 15 leggyakoribb foglalat típusnál')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Átlagos árak a leggyakoribb socket-eknél:")
display(df_top_sockets_price.groupby('socket')['price'].mean().sort_values(ascending=False))

In [ ]:
# Új könyvtárak importálása (ezeket a notebook elejéhez is teheted)


# === Finding best Price regression ===
# Adatok előkészítése: itt a df_clean már a tisztított formátumú
df_price_base = df_clean.dropna(subset=['price'] + features_price).copy()
print(df_price_base.info())
results_list_price = []

# --- 1. Modell eredmények (eredeti adatokon + Socket) ---
y_orig_price = df_price_base["price"]
X_orig_price = pd.get_dummies(df_price_base[features_price], columns=["category", "socket_grouped"])
X_train_orig_price, X_test_orig_price, y_train_orig_price, y_test_orig_price = train_test_split(X_orig_price, y_orig_price, test_size=0.2, random_state=42)

# Kibővített modell lista (XGBoost, CatBoost és egy RobustScaler-el ellátott SVR)
models_orig_price = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=10.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest_Original": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost_Original": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42),
    "CatBoost_Original": CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6, random_seed=42, verbose=0),
    "SVR_Robust_Original": make_pipeline(RobustScaler(), SVR(kernel='rbf', C=100.0, gamma='scale')),
    "HistGradientBoosting_Original": HistGradientBoostingRegressor(loss='poisson', random_state=42),
    "PolinomialRidge_original": make_pipeline(PolynomialFeatures(degree=2), StandardScaler(), Ridge(alpha=10.0)),
    "TweedieRegression_Original": make_pipeline(StandardScaler(), TweedieRegressor(power=1.5, link='log')),
    "KNN_Original": make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=5)),
}

print("--- Modell eredmények (eredeti adatokon - Price) ---")
for name, model in models_orig_price.items():
    model.fit(X_train_orig_price, y_train_orig_price)
    y_pred = model.predict(X_test_orig_price)
    rmse = np.sqrt(mean_squared_error(y_test_orig_price, y_pred))
    mae = mean_absolute_error(y_test_orig_price, y_pred)
    r2 = r2_score(y_test_orig_price, y_pred)
    results_list_price.append({'Modell': f"{name}", 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2})

# --- 2. Modell eredmények (logaritmikus transzformációval + Socket) ---
y_orig_log_price = np.log1p(y_orig_price)
X_train_orig_log_price, X_test_orig_log_price, y_train_orig_log_price, y_test_orig_log_price = train_test_split(X_orig_price, y_orig_log_price, test_size=0.2, random_state=42)

# Ugyanezek a modellek bevetése a logaritmizált célváltozón is
models_log_orig_price = {
    "Linear_LogTransformed": LinearRegression(),
    "Ridge_LogTransformed": Ridge(alpha=10.0),
    "Lasso_LogTransformed": Lasso(alpha=0.1),
    "ElasticNet_LogTransformed": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest_LogTransformed": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost_LogTransformed": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42),
    "CatBoost_LogTransformed": CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6, random_seed=42, verbose=0),
    "SVR_Robust_LogTransformed": make_pipeline(RobustScaler(), SVR(kernel='rbf', C=10.0, gamma='scale')),
    "HistGradientBoosting_LogTransformed": HistGradientBoostingRegressor(loss='poisson', random_state=42),
    "PolinomialRidge_LogTransformed": make_pipeline(PolynomialFeatures(degree=2), StandardScaler(), Ridge(alpha=10.0)),
}

print("\n--- Modell eredmények (log transzformáció - Price) ---")
for name, model in models_log_orig_price.items():
    model.fit(X_train_orig_log_price, y_train_orig_log_price)
    y_pred_log = model.predict(X_test_orig_log_price)
    y_pred = np.expm1(y_pred_log)
    rmse = np.sqrt(mean_squared_error(y_test_orig_price, y_pred))
    mae = mean_absolute_error(y_test_orig_price, y_pred)
    r2 = r2_score(y_test_orig_price, y_pred)
    results_list_price.append({'Modell': f"{name}", 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2})

# --- 3. Modell eredmények (outlierek kezelése IQR-rel) ---
Q1_price = df_price_base['price'].quantile(0.25)
Q3_price = df_price_base['price'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound_price = Q1_price - 1.5 * IQR_price
upper_bound_price = Q3_price + 1.5 * IQR_price

df_filtered_price = df_price_base[(df_price_base['price'] >= lower_bound_price) & (df_price_base['price'] <= upper_bound_price)].copy()

y_filt_price = df_filtered_price["price"]
X_filt_price = pd.get_dummies(df_filtered_price[features_price], columns=["category", "socket_grouped"])
X_train_filt_price, X_test_filt_price, y_train_filt_price, y_test_filt_price = train_test_split(X_filt_price, y_filt_price, test_size=0.2, random_state=42)

models_filtered_price = {
    "Linear_IQR": LinearRegression(),
    "RandomForest_IQR": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost_IQR": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
}

print("\n--- Modell eredmények (outlierek kezelése IQR-rel - Price) ---")
for name, model in models_filtered_price.items():
    model.fit(X_train_filt_price, y_train_filt_price)
    y_pred_filt = model.predict(X_test_filt_price)
    rmse_filt = np.sqrt(mean_squared_error(y_test_filt_price, y_pred_filt))
    mae_filt = mean_absolute_error(y_test_filt_price, y_pred_filt)
    r2_filt = r2_score(y_test_filt_price, y_pred_filt)
    results_list_price.append({'Modell': f"{name}", 'RMSE': rmse_filt, 'MAE': mae_filt, 'R2 Score': r2_filt})

# Eredmények összegzése
comparison_df_price = pd.DataFrame(results_list_price).sort_values(by="R2 Score", ascending=False)
display(comparison_df_price)

In [ ]:
# === 2. LÉPCSŐ: Price Predikciós Modell ===

# 1. TANÍTÁS: Szigorúan az eredeti, valós Price-szal ÉS valós TDP-vel rendelkező adatokon!
# Itt df_clean-et használunk, NEM a df_cascade-et!
df_price_train = df_clean.dropna(subset=['price'] + features_price).copy()

# Logaritmizáljuk az árat!
y_price_train_log = np.log1p(df_price_train['price'])
X_price_train = pd.get_dummies(df_price_train[features_price], columns=['category', 'socket_grouped'])

# Price Modell betanítása a tiszta adatokon
price_model = RandomForestRegressor(n_estimators=100, random_state=42)
price_model.fit(X_price_train, y_price_train_log)


df_cascade = df_clean.copy()
# 2. PÓTLÁS (Imputation): Ahol az eredeti adatbázisban (df_cascade) hiányzott az ár.
missing_price_mask = df_cascade['price'].isnull()
df_missing_price = df_cascade[missing_price_mask].copy()

if not df_missing_price.empty:
    X_missing_price = pd.get_dummies(df_missing_price[features_price], columns=['category', 'socket_grouped'])
    X_missing_price = X_missing_price.reindex(columns=X_price_train.columns, fill_value=0)
    
    # Becsült Árak beírása a cascade adatbázisba
    df_cascade.loc[missing_price_mask, 'price'] = np.round(np.expm1(price_model.predict(X_missing_price)), 2)
    df_cascade['price_is_imputed'] = missing_price_mask.astype(int)

# Opcionális: Származtatott értékek (cpuValue, threadValue) újraszámolása a már teli adatbázisban
df_cascade['cpuValue'] = df_cascade['cpuMark'] / df_cascade['price']
df_cascade['threadValue'] = df_cascade['threadMark'] / df_cascade['price']

print(f"2. LÉPCSŐ KÉSZ: Hiányzó Price értékek pótolva {missing_price_mask.sum()} sorban.")

df_cascade.to_csv('../data/processed/CPU_benchmark_v4_price_imputed.csv', index=False)
print('Kiegészített adathalmaz elmentve!')

In [ ]:
# === Finding best Power Performance regression ===

# Adatok előkészítése: A már imputált (árral kiegészített) adatbázist használjuk
# Csak azokat a sorokat nézzük, ahol a powerPerf NEM hiányzik (ez a gold standard tanítóhalmaz)



df_pp_base = df_cascade.dropna(subset=['powerPerf'] + features_pp).copy()
# print(df_pp_base.info())
results_list_pp = []

# Itt a features_pp globális listát használjuk (amiben benne van a pótolt price is!)
y_orig_pp = df_pp_base["powerPerf"]
X_orig_pp = pd.get_dummies(df_pp_base[features_pp], columns=["category", "socket_grouped"])

# --- 1. Modell eredmények (eredeti adatokon) ---
X_train_orig_pp, X_test_orig_pp, y_train_orig_pp, y_test_orig_pp = train_test_split(X_orig_pp, y_orig_pp, test_size=0.2, random_state=42)

base_estimators = [
    ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)),
    ('cat', CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6, random_seed=42, verbose=0)),
    
    # RF helyett egy teljesen más logika: KNN (a RobustScaler kötelező elé!)
    ('knn', make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=5))),
    
    ('svr', make_pipeline(RobustScaler(), SVR(kernel='rbf', C=100.0, gamma='scale')))
]
stacking_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=Lasso(alpha=0.1), # Meta-modell csere Lasso-ra, hogy kiszűrje a gyenge prediktorokat
    cv=5,
    n_jobs=-1, # Hogy mindegyik CPU magot használja, gyorsabb legyen a futás
    passthrough=True # A meta-modell látja az eredeti oszlopokat is
)

base_estimators2 = [
    ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)),
    ('cat', CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6, random_seed=42, verbose=0)),
    
    # RF helyett egy teljesen más logika: KNN (a RobustScaler kötelező elé!)
    ('knn', make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=5))),
    
    ('svr', make_pipeline(RobustScaler(), SVR(kernel='rbf', C=100.0, gamma='scale')))
]

stacking_model2 = StackingRegressor(
    estimators=base_estimators2,
    final_estimator=ElasticNet(alpha=0.1, l1_ratio=0.5), # Meta-modell csere ElasticNet-re, hogy kiszűrje a gyenge prediktorokat és kezelje a multikollinearitást
    cv=5,
    n_jobs=-1, # Hogy mindegyik CPU magot használja, gyorsabb legyen a futás
    passthrough=True # A meta-modell látja az eredeti oszlopokat is
)

models_orig_pp = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=10.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest_Original": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost_Original": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42),
    "CatBoost_Original": CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6, random_seed=42, verbose=0),
    "SVR_Robust_Original": make_pipeline(RobustScaler(), SVR(kernel='rbf', C=100.0, gamma='scale')),
    "HistGradientBoosting_Original": HistGradientBoostingRegressor(loss='poisson', random_state=42),
    "PolinomialRidge_original": make_pipeline(PolynomialFeatures(degree=2), StandardScaler(), Ridge(alpha=10.0)),
    "TweedieRegression_Original": make_pipeline(StandardScaler(), TweedieRegressor(power=1.5, link='log')),
    "KNN_Original": make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=5)),
    "Stacking_Original": stacking_model,
    "Stacking2_Original": stacking_model2,
}

print("--- Modell eredmények (eredeti adatokon - PowerPerf) ---")
for name, model in models_orig_pp.items():
    model.fit(X_train_orig_pp, y_train_orig_pp)
    y_pred = model.predict(X_test_orig_pp)
    rmse = np.sqrt(mean_squared_error(y_test_orig_pp, y_pred))
    mae = mean_absolute_error(y_test_orig_pp, y_pred)
    r2 = r2_score(y_test_orig_pp, y_pred)
    results_list_pp.append({'Modell': f"{name}", 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2})

# --- 2. Modell eredmények (logaritmikus transzformációval) ---
# Biztonságos logaritmus (log1p), ha lennének 0 közeli értékek
y_orig_log_pp = np.log1p(y_orig_pp)
X_train_orig_log_pp, X_test_orig_log_pp, y_train_orig_log_pp, y_test_orig_log_pp = train_test_split(X_orig_pp, y_orig_log_pp, test_size=0.2, random_state=42)

models_log_orig_pp = {
    "Linear_LogTransformed": LinearRegression(),
    "Ridge_LogTransformed": Ridge(alpha=10.0),
    "Lasso_LogTransformed": Lasso(alpha=0.1),
    "ElasticNet_LogTransformed": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest_LogTransformed": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost_LogTransformed": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42),
    "CatBoost_LogTransformed": CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6, random_seed=42, verbose=0),
    "SVR_Robust_LogTransformed": make_pipeline(RobustScaler(), SVR(kernel='rbf', C=10.0, gamma='scale')),
    "HistGradientBoosting_LogTransformed": HistGradientBoostingRegressor(loss='poisson', random_state=42),
    "PolinomialRidge_LogTransformed": make_pipeline(PolynomialFeatures(degree=2), StandardScaler(), Ridge(alpha=10.0)),
}

print("\n--- Modell eredmények (log transzformáció - PowerPerf) ---")
for name, model in models_log_orig_pp.items():
    model.fit(X_train_orig_log_pp, y_train_orig_log_pp)
    y_pred_log = model.predict(X_test_orig_log_pp)
    y_pred = np.expm1(y_pred_log) # Visszaalakítás a valós skálára
    rmse = np.sqrt(mean_squared_error(y_test_orig_pp, y_pred))
    mae = mean_absolute_error(y_test_orig_pp, y_pred)
    r2 = r2_score(y_test_orig_pp, y_pred)
    results_list_pp.append({'Modell': f"{name}", 'RMSE': rmse, 'MAE': mae, 'R2 Score': r2})

# --- 3. Modell eredmények (outlierek kezelése IQR-rel) ---
X_train_iqr, X_test_iqr, y_train_iqr, y_test_iqr = train_test_split(
    X_orig_pp, y_orig_pp, test_size=0.2, random_state=42
)

Q1_pp = df_pp_base['powerPerf'].quantile(0.25)
Q3_pp = df_pp_base['powerPerf'].quantile(0.75)
IQR_pp = Q3_pp - Q1_pp
lower_bound_pp = Q1_pp - 1.5 * IQR_pp
upper_bound_pp = Q3_pp + 1.5 * IQR_pp

train_mask = (y_train_iqr >= lower_bound_pp) & (y_train_iqr <= upper_bound_pp)
X_train_filt_pp = X_train_iqr[train_mask]
y_train_filt_pp = y_train_iqr[train_mask]

models_filtered_pp = {
    "Linear_IQR": LinearRegression(),
    "Ridge_IQR": Ridge(alpha=10.0),
    "Lasso_IQR": Lasso(alpha=0.1),
    "ElasticNet_IQR": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RandomForest_IQR": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost_IQR": XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42),
    "CatBoost_IQR": CatBoostRegressor(iterations=200, learning_rate=0.05, depth=6, random_seed=42, verbose=0),
    "SVR_Robust_IQR": make_pipeline(RobustScaler(), SVR(kernel='rbf', C=100.0, gamma='scale'))
}

print("\n--- Modell eredmények (outlierek kezelése IQR-rel - PowerPerf) ---")
print(f"Eredeti tanító sorok: {len(X_train_iqr)}")
print(f"Szűrt tanító sorok: {len(X_train_filt_pp)} (Ezen tanul a modell)")
print(f"Teszthalmaz mérete: {len(X_test_iqr)} (Érintetlen, valós adatok)\n")

for name, model in models_filtered_pp.items():
    model.fit(X_train_filt_pp, y_train_filt_pp) # Tanítás a szűrt adatokon
    y_pred_filt = model.predict(X_test_iqr)     # Tesztelés a SZŰRETLEN adatokon!
    
    rmse_filt = np.sqrt(mean_squared_error(y_test_iqr, y_pred_filt))
    mae_filt = mean_absolute_error(y_test_iqr, y_pred_filt)
    r2_filt = r2_score(y_test_iqr, y_pred_filt)
    
    # Hozzáadjuk a listához (ügyelve, hogy ne írjuk felül a korábbiakat)
    results_list_pp.append({'Modell': f"{name}_Corrected", 'RMSE': rmse_filt, 'MAE': mae_filt, 'R2 Score': r2_filt})

# --- Eredmények összefoglalása ---
comparison_df_pp = pd.DataFrame(results_list_pp).sort_values(by="R2 Score", ascending=False)
display(comparison_df_pp)

In [ ]:
# === 3. LÉPCSŐ: Power Performance Predikciós Modell (Fő feladat) ===

# 1. TANÍTÁS: Valós PowerPerf és Valós Price megléte esetén (Eredeti df_cascade-ből származtatva)
# (A df_cascade-ben ha van powerPerf, akkor van TDP is)
df_pp_train = df_cascade.dropna(subset=['powerPerf'] + features_pp).copy()

# Logaritmizáljuk a hatékonyságot!
y_pp_train = df_pp_train['powerPerf']
X_pp_train = pd.get_dummies(df_pp_train[features_pp], columns=['category', 'socket_grouped'])

pp_model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
pp_model.fit(X_pp_train, y_pp_train)

In [ ]:
# === Betanított modell és oszlopstruktúra mentése a webapphoz ===


# Mappa létrehozása, ha még nem létezne
os.makedirs('../models', exist_ok=True)

# 2. Price (Ár) modell és oszlopok mentése
joblib.dump(price_model, '../models/price_model.pkl')
joblib.dump(list(X_price_train.columns), '../models/price_model_columns.pkl')

# 3. Power Performance modell és oszlopok mentése
joblib.dump(pp_model, '../models/powerPerf_model.pkl')
joblib.dump(list(X_pp_train.columns), '../models/powerPerf_model_columns.pkl')

# Mentsük ki a kategóriákat és a top socketeket is, amiket a modell ismer
joblib.dump(list(df_cascade['category'].unique()), '../models/categories_list.pkl')
# Fontos, hogy az 'Other' is benne legyen a listában a webapp számára
top_sockets_with_other = list(top_20_sockets)
if 'Other' not in top_sockets_with_other:
    top_sockets_with_other.append('Other')
joblib.dump(top_sockets_with_other, '../models/sockets_list.pkl')

print("Minden modell (Price, PowerPerf) és oszlopstruktúra sikeresen elmentve a ../models mappába!")

In [ ]:
# === TELJESÍTMÉNY ANALÍZIS ÉS VIZUALIZÁCIÓ ===

# 1. Adatok szétválasztása teszteléshez (mivel a pp_model a teljes X_pp_train-en lett tanítva)
# Csinálunk egy gyors split-et csak a vizualizáció kedvéért
X_train_v, X_test_v, y_train_v_log, y_test_v_log = train_test_split(
    X_pp_train, y_pp_train, test_size=0.2, random_state=42
)

# Predikció a logaritmikus teszthalmazon
y_pred_log = pp_model.predict(X_test_v)

# VISSZAALAKÍTÁS a valós skálára (Log -> Valós érték)
y_test_actual = y_test_v_log
y_pred_actual = y_pred_log

sns.set_theme(style="whitegrid")

# --- A) Valós vs. Becsült értékek ---
plt.figure(figsize=(10, 6))
plt.scatter(y_test_actual, y_pred_actual, alpha=0.5, color='dodgerblue', edgecolor='k')
max_val = max(y_test_actual.max(), y_pred_actual.max())
plt.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='Tökéletes illeszkedés')
plt.title('Power Performance: Valós vs. Becsült (Visszaalakított skálán)')
plt.xlabel('Valós Power Performance')
plt.ylabel('Becsült Power Performance')
plt.legend()
plt.show()

# --- B) Hibák (Reziduumok) eloszlása ---
residuals = y_test_actual - y_pred_actual
plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True, color='purple')
plt.axvline(0, color='red', linestyle='--')
plt.title('Predikciós hibák eloszlása (Watt/Teljesítmény különbség)')
plt.xlabel('Hiba mértéke')
plt.show()

# --- C) Feature Importance (Top 15) ---
importance = pd.DataFrame({
    'Feature': X_pp_train.columns,
    'Importance': pp_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=importance.head(15), x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title('Top 15 legfontosabb bemeneti változó a Power Performance becslésénél')
plt.show()